# Inspect DuckDB

Tables, schema, row counts, symbols, date ranges. Lives in **`research/db/`** with the refresh scripts.

Default DB: **`data/research_roostoo_1m.duckdb`**. Repo root is auto-detected from cwd.

In [12]:
import sys
import subprocess

try:
    import duckdb
except ModuleNotFoundError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "duckdb", "pandas"])
    import duckdb

import pandas as pd
from pathlib import Path


def repo_root() -> Path:
    p = Path.cwd().resolve()
    for _ in range(10):
        if (p / "research" / "db" / "inspect_duckdb.ipynb").exists() and (p / "data").is_dir():
            return p
        if p.parent == p:
            break
        p = p.parent
    raise RuntimeError(f"Could not find repo root. cwd={Path.cwd().resolve()}")


ROOT = repo_root()
CANDIDATES = [
    ROOT / "data" / "research_roostoo_1m.duckdb",
    ROOT / "data" / "research_1m.duckdb",
    ROOT / "data" / "research.duckdb",
]
DB_PATH = next((p for p in CANDIDATES if p.exists()), None)
if DB_PATH is None:
    raise FileNotFoundError(
        "No DuckDB found. Tried:\n  " + "\n  ".join(str(p) for p in CANDIDATES)
        + "\n\nRefresh: see research/db/README.md"
    )

print("Repo:  ", ROOT)
print("Using: ", DB_PATH)

Repo:   /Users/ankoo/Desktop/Projects/Margin Call Capital 2.0/margincallcapital
Using:  /Users/ankoo/Desktop/Projects/Margin Call Capital 2.0/margincallcapital/data/research_roostoo_1m.duckdb


In [13]:
con = duckdb.connect(str(DB_PATH), read_only=True)

tables = con.execute(
    "SELECT table_schema, table_name FROM information_schema.tables "
    "WHERE table_schema NOT IN ('information_schema', 'pg_catalog') ORDER BY 1, 2"
).df()
display(tables)

,table_schema,table_name
0,main,ohlcv
1,main,roostoo_load_meta


In [14]:
display(con.execute("DESCRIBE ohlcv").df())

,column_name,column_type,null,key,default,extra
0,ts,TIMESTAMP,YES,None,None,None
1,open,DOUBLE,YES,None,None,None
2,high,DOUBLE,YES,None,None,None
3,low,DOUBLE,YES,None,None,None
4,close,DOUBLE,YES,None,None,None
5,volume,DOUBLE,YES,None,None,None
6,symbol,VARCHAR,YES,None,None,None
7,interval,VARCHAR,YES,None,None,None


In [15]:
display(
    con.execute(
        "SELECT interval, COUNT(*)::BIGINT AS n_rows FROM ohlcv GROUP BY 1 ORDER BY 1"
    ).df()
)

,interval,n_rows
0,15m,455366
1,1h,114683
2,1m,6782025
3,5m,1360389


In [16]:
# Global min/max timestamp by interval (spot stale cutoffs)
display(
    con.execute(
        """
        SELECT interval,
               MIN(ts) AS min_ts,
               MAX(ts) AS max_ts,
               COUNT(*)::BIGINT AS n_rows
        FROM ohlcv
        GROUP BY 1
        ORDER BY 1
        """
    ).df()
)

,interval,min_ts,max_ts,n_rows
0,15m,2025-12-19 17:30:00,2026-03-19 17:15:00,455366
1,1h,2025-12-19 17:00:00,2026-03-19 17:00:00,114683
2,1m,2025-12-19 17:33:00,2026-03-19 17:26:00,6782025
3,5m,2025-12-19 17:30:00,2026-03-19 17:25:00,1360389


In [17]:
by_iv = con.execute(
    "SELECT interval, COUNT(DISTINCT symbol)::BIGINT AS n_symbols FROM ohlcv GROUP BY 1 ORDER BY 1"
).df()
display(by_iv)
nu = con.execute("SELECT COUNT(DISTINCT symbol)::BIGINT AS n FROM ohlcv").fetchone()[0]
print(f"Distinct symbols (any interval): {nu}")

,interval,n_symbols
0,15m,67
1,1h,67
2,1m,67
3,5m,67


Distinct symbols (any interval): 67


In [18]:
sym_1h = con.execute(
    "SELECT DISTINCT symbol FROM ohlcv WHERE interval = '1h' ORDER BY symbol"
).df()["symbol"].tolist()
print(f"1h symbols: {len(sym_1h)}")
sym_1h

1h symbols: 67


['1000CHEEMS-USD',
 'AAVE-USD',
 'ADA-USD',
 'APT-USD',
 'ARB-USD',
 'ASTER-USD',
 'AVAX-USD',
 'AVNT-USD',
 'BIO-USD',
 'BMT-USD',
 'BNB-USD',
 'BONK-USD',
 'BTC-USD',
 'CAKE-USD',
 'CFX-USD',
 'CRV-USD',
 'DOGE-USD',
 'DOT-USD',
 'EDEN-USD',
 'EIGEN-USD',
 'ENA-USD',
 'ETH-USD',
 'FET-USD',
 'FIL-USD',
 'FLOKI-USD',
 'FORM-USD',
 'HBAR-USD',
 'HEMI-USD',
 'ICP-USD',
 'LINEA-USD',
 'LINK-USD',
 'LISTA-USD',
 'LTC-USD',
 'MIRA-USD',
 'NEAR-USD',
 'OMNI-USD',
 'ONDO-USD',
 'OPEN-USD',
 'PAXG-USD',
 'PENDLE-USD',
 'PENGU-USD',
 'PEPE-USD',
 'PLUME-USD',
 'POL-USD',
 'PUMP-USD',
 'S-USD',
 'SEI-USD',
 'SHIB-USD',
 'SOL-USD',
 'SOMI-USD',
 'STO-USD',
 'SUI-USD',
 'TAO-USD',
 'TON-USD',
 'TRUMP-USD',
 'TRX-USD',
 'TUT-USD',
 'UNI-USD',
 'VIRTUAL-USD',
 'WIF-USD',
 'WLD-USD',
 'WLFI-USD',
 'XLM-USD',
 'XPL-USD',
 'XRP-USD',
 'ZEC-USD',
 'ZEN-USD']

In [19]:
display(
    con.execute(
        """
        SELECT symbol,
               COUNT(*)::BIGINT AS n,
               MIN(ts) AS min_ts,
               MAX(ts) AS max_ts
        FROM ohlcv
        WHERE interval = '1h'
        GROUP BY 1
        ORDER BY max_ts ASC
        LIMIT 15
        """
    ).df()
)
print("(sorted by max_ts ascending — lagging symbols first)")

,symbol,n,min_ts,max_ts
0,EIGEN-USD,1711,2025-12-19 17:00:00,2026-02-28 23:00:00
1,LINEA-USD,1711,2025-12-19 17:00:00,2026-02-28 23:00:00
2,LINK-USD,1711,2025-12-19 17:00:00,2026-02-28 23:00:00
3,AVAX-USD,1711,2025-12-19 17:00:00,2026-02-28 23:00:00
4,AAVE-USD,1711,2025-12-19 17:00:00,2026-02-28 23:00:00
5,PLUME-USD,1711,2025-12-19 17:00:00,2026-02-28 23:00:00
6,ZEN-USD,1711,2025-12-19 17:00:00,2026-02-28 23:00:00
7,MIRA-USD,1711,2025-12-19 17:00:00,2026-02-28 23:00:00
8,CFX-USD,1711,2025-12-19 17:00:00,2026-02-28 23:00:00
9,FORM-USD,1711,2025-12-19 17:00:00,2026-02-28 23:00:00


(sorted by max_ts ascending — lagging symbols first)


In [20]:
_iv = "1h"
display(
    con.execute(
        "SELECT * FROM ohlcv WHERE interval = ? ORDER BY ts DESC, symbol LIMIT 15",
        [_iv],
    ).df()
)

,ts,open,high,low,close,volume,symbol,interval
0,2026-03-19 17:00:00,0.756,0.762,0.756,0.762,1563.87,OMNI-USD,1h
1,2026-03-19 15:00:00,0.758,0.758,0.734,0.734,3119.68,OMNI-USD,1h
2,2026-03-19 14:00:00,0.755,0.768,0.755,0.768,2101.67,OMNI-USD,1h
3,2026-03-19 13:00:00,0.730,0.768,0.730,0.768,2062.89,OMNI-USD,1h
4,2026-03-19 07:00:00,0.731,0.739,0.731,0.739,260.51,OMNI-USD,1h
5,2026-03-19 06:00:00,0.739,0.740,0.739,0.740,243.34,OMNI-USD,1h
6,2026-03-19 03:00:00,0.740,0.740,0.739,0.739,243.51,OMNI-USD,1h
7,2026-03-19 01:00:00,0.742,0.742,0.742,0.742,9.66,OMNI-USD,1h
8,2026-03-19 00:00:00,0.738,0.738,0.738,0.738,151.67,OMNI-USD,1h
9,2026-03-18 21:00:00,0.741,0.748,0.727,0.735,4815.61,OMNI-USD,1h


In [21]:
try:
    display(con.execute("SELECT * FROM roostoo_load_meta ORDER BY base").df())
except Exception as e:
    print("No roostoo_load_meta table:", e)

,base,pair,source,rows_1m,rows_5s,updated_at
0,1000CHEEMS,1000CHEEMS/USD,binance_vision,102627,0,2026-03-20 01:32:57.747496
1,AAVE,AAVE/USD,binance_vision,102627,0,2026-03-20 01:32:59.628067
2,ADA,ADA/USD,binance_vision,102627,0,2026-03-20 01:33:01.197360
3,APT,APT/USD,binance_vision,102627,0,2026-03-20 01:33:02.574197
4,ARB,ARB/USD,binance_vision,102627,0,2026-03-20 01:33:04.389626
...,...,...,...,...,...,...
62,XLM,XLM/USD,binance_vision,102627,0,2026-03-20 01:39:50.114320
63,XPL,XPL/USD,binance_vision,102627,0,2026-03-20 01:39:52.012155
64,XRP,XRP/USD,binance_vision,102627,0,2026-03-20 01:39:53.842889
65,ZEC,ZEC/USD,binance_vision,102627,0,2026-03-20 01:39:55.810908


In [22]:
query = """
SELECT interval, symbol, COUNT(*) AS n
FROM ohlcv
GROUP BY 1, 2
ORDER BY n ASC
LIMIT 20
"""
display(con.execute(query).df())

,interval,symbol,n
0,1h,SHIB-USD,1711
1,1h,ADA-USD,1711
2,1h,1000CHEEMS-USD,1711
3,1h,WLD-USD,1711
4,1h,XLM-USD,1711
5,1h,SOMI-USD,1711
6,1h,OPEN-USD,1711
7,1h,XPL-USD,1711
8,1h,HBAR-USD,1711
9,1h,UNI-USD,1711
